# Phase 4: Untuned Base Model Zero-Shot Baseline Evaluation

This notebook evaluates the untuned base model (`Qwen/Qwen2.5-3B-Instruct`) without any fine-tuning (`--adapter none`) on:
1. `b77_test` (3,080 test queries)
2. `amb_test` (200 held-out ambiguous queries)

This provides the zero-shot baseline to compare against our reasoning-distilled fine-tuned SLM.

In [ ]:
import os, sys, math, time, json, random
from pathlib import Path
import torch

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("WARNING: No GPU detected!")

In [ ]:
!pip install -q "transformers>=4.45.0" "bitsandbytes>=0.43.0" "accelerate>=0.34.0" pandas

In [ ]:
from pathlib import Path

input_root = Path("/kaggle/input")
print("Scanning /kaggle/input for dataset...")

data_dir = None
candidates_data = [
    input_root / "banking77-eval-data",
    input_root / "datasets/goofybhatoora/banking77-eval-data",
    input_root / "banking77-eval-data/eval_upload"
]
for c in candidates_data:
    if (c / "test.csv").exists():
        data_dir = c
        break

if not data_dir:
    for p in input_root.glob("**/test.csv"):
        data_dir = p.parent
        break

print(f"Using Data Dir: {data_dir}")
assert data_dir and (data_dir / "test.csv").exists(), "Could not locate test.csv!"

In [ ]:
import math
import time
import json
import random
from pathlib import Path
import pandas as pd
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

BASE_MODEL = "Qwen/Qwen2.5-3B-Instruct"
PREFILL = {"free": "", "fast": "<fast>", "think": "<think>"}

def load_jsonl(path: Path) -> list[dict]:
    with open(path, encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

def system_prompt(data_dir: Path) -> str:
    with open(data_dir / "train.jsonl", encoding="utf-8") as handle:
        return json.loads(handle.readline())["messages"][0]["content"]

def build_sets(data_dir: Path, limit: int | None = None) -> dict[str, list[dict]]:
    test = pd.read_csv(data_dir / "test.csv")
    sets = {"b77_test": [{"id": f"test-{i:04d}", "text": t, "gold": c, "candidates_gold": None}
                         for i, (t, c) in enumerate(zip(test["text"], test["category"]))]
    }
    amb = load_jsonl(data_dir / "ambiguous_test.jsonl")
    sets["amb_test"] = [{"id": r["id"], "text": r["text"], "gold": None, "candidates_gold": r["candidates"],
                         "family": r["family"]} for r in amb]
    if limit:
        sets = {name: rows[:limit] for name, rows in sets.items()}
    return sets

def run_eval(
    data_dir: Path,
    out_dir: Path,
    modes: list[str] = ["free"],
    sets: list[str] = ["b77_test", "amb_test"],
    batch_size: int = 16,
    max_new_tokens: int = 220,
    limit: int | None = None
):
    out_dir.mkdir(parents=True, exist_ok=True)
    random.seed(77)
    torch.manual_seed(77)

    print(f"\n--- Loading Tokenizer: {BASE_MODEL} ---")
    tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, padding_side="left")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print(f"--- Loading Base Model (Zero-Shot 4-bit NF4): {BASE_MODEL} ---")
    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        quantization_config=BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.float16
        ),
        device_map={"": 0},
    )
    model.eval()

    system = system_prompt(data_dir)
    eval_sets = build_sets(data_dir, limit)

    for set_name in sets:
        rows = eval_sets[set_name]
        for mode in modes:
            out_path = out_dir / f"{set_name}__{mode}.jsonl"
            if out_path.exists():
                print(f"Skip {out_path.name} (already exists)")
                continue
            
            print(f"\nGenerating {set_name} in '{mode}' mode ({len(rows)} queries)...")
            prompts = [
                tokenizer.apply_chat_template(
                    [{"role": "system", "content": system}, {"role": "user", "content": row["text"].strip()}],
                    tokenize=False, add_generation_prompt=True) + PREFILL[mode]
                for row in rows
            ]
            order = sorted(range(len(rows)), key=lambda i: len(prompts[i]))
            results: dict[int, dict] = {}
            t_start = time.time()

            for start in range(0, len(order), batch_size):
                batch_idx = order[start:start + batch_size]
                encoded = tokenizer([prompts[i] for i in batch_idx], return_tensors="pt", padding=True).to(model.device)
                begin = time.time()
                with torch.no_grad():
                    generated = model.generate(
                        **encoded, max_new_tokens=max_new_tokens, do_sample=False,
                        output_scores=True, return_dict_in_generate=True, pad_token_id=tokenizer.pad_token_id)
                seconds = (time.time() - begin) / len(batch_idx)

                scores = model.compute_transition_scores(generated.sequences, generated.scores, normalize_logits=True)
                new_tokens = generated.sequences[:, encoded["input_ids"].shape[1]:]

                for row_pos, i in enumerate(batch_idx):
                    ids = new_tokens[row_pos].tolist()
                    keep = [t for t in range(len(ids)) if ids[t] != tokenizer.pad_token_id]
                    token_logprobs = []
                    for t in keep:
                        val = float(scores[row_pos, t])
                        if math.isinf(val) or math.isnan(val):
                            val = -999.0
                        token_logprobs.append([tokenizer.decode(ids[t]), round(val, 4)])
                    
                    text = tokenizer.decode([ids[t] for t in keep], skip_special_tokens=True)
                    results[i] = {
                        **rows[i],
                        "mode": mode,
                        "output": PREFILL[mode] + text,
                        "n_new_tokens": len(keep),
                        "token_logprobs": token_logprobs,
                        "seconds_per_query": round(seconds, 4)
                    }

                if (start // batch_size) % 15 == 0 or (start + batch_size >= len(order)):
                    done = min(start + batch_size, len(order))
                    elapsed = time.time() - t_start
                    speed = done / max(elapsed, 0.01)
                    print(f"  {set_name}/{mode}: {done}/{len(order)} queries done ({speed:.1f} q/s)", flush=True)

            with open(out_path, "w", encoding="utf-8") as handle:
                for i in range(len(rows)):
                    handle.write(json.dumps(results[i], ensure_ascii=False) + "\n")
            print(f"--> Wrote {out_path} ({len(rows)} records)")

    print("\n--- Base Model Evaluation Complete! ---")

In [ ]:
# Execute evaluation for untuned base model
eval_out_dir = Path("/kaggle/working/eval_base")
run_eval(
    data_dir=data_dir,
    out_dir=eval_out_dir,
    modes=["free"],
    sets=["b77_test", "amb_test"],
    batch_size=16,
    max_new_tokens=220
)

In [ ]:
# Verification of all generated base model files
print("\n--- Output Verification ---")
eval_out_dir = Path("/kaggle/working/eval_base")
for f in sorted(eval_out_dir.glob("*.jsonl")):
    num_lines = sum(1 for _ in open(f, encoding="utf-8"))
    size_kb = f.stat().st_size / 1024
    print(f"{f.name:30s} -> {num_lines:5d} records ({size_kb:8.1f} KB)")